# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Lane 2, Refresh scoring. The model from Week 5 produces a number. This notebook turns that number
into something an editor can work: a ranked queue where every row carries a reason and a next
action, plus the limits, the review rules, and the triggers that say when to stop trusting it.

One thing to keep straight throughout. The **ranking** comes from the leakage-safe model, which is
only allowed to see static page properties and the prior 30-day window. The **reason codes** are
descriptive diagnostics read off the page's current state, including CTR and position, which the
model itself may not use as inputs. Ranking is a prediction and has to be clean. A reason code is a
note to a human about what to look at, and it is allowed to use what is on the page today.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue is built in two steps. The model scores every visible page for decline risk and sorts
them. Then a small deterministic rule set reads each page's current state and attaches the reason
it is on the list, plus the action that follows from that reason. The rules are thresholds on
observed columns, not a second model, so an editor can check any row by hand.

| Reason code | Condition | Action |
|---|---|---|
| `low_ctr_for_position` | ranks in top 3, page 1 or striking distance, but CTR sits below the median for its tier, and it still earns real prior-window impressions | rewrite title and meta |
| `stale_but_earning` | not updated in the longest quarter of the corpus, yet still earning real prior-window impressions | refresh the content |
| `thin_for_demand` | in the shortest quarter by word count while earning real impressions | expand the page |
| `low_visibility` | sits on page 3 to 5 or deeper | relevance work and internal links |
| `no_clear_lever` | none of the above fires | watch only, do not spend a slot |

Order matters: the rules are checked top to bottom and the first match wins, so a page that is both
under-clicked and stale is sent to the snippet fix, which is the cheaper intervention.

**Archetype to action.** Reason codes describe the symptom on one page. Archetypes group pages by
their structural profile, which is what a content lead plans capacity around. I build them from age
and freshness, two columns the model is allowed to see and a human can read instantly.

| Archetype | Profile | Default action | Why |
|---|---|---|---|
| Fresh starter | under 90 days old | leave alone, let it mature | too early to read a trend from a 30-day comparison |
| Maturing asset | 90 days to a year, updated recently enough | first call on snippet and refresh work | the window where decline is most common here, 0.69 at 91 to 180 days |
| Neglected earner | untouched for over a quarter, still earning | refresh the content | the clearest refresh case |
| Old survivor | over a year old, still visible | protect, low priority | lowest decline share here at 0.43, and already past the risky window |
| Low-demand tail | deep positions, little traffic | do not queue | a refresh slot spent here buys almost nothing |

**The decay and refresh insight.** The numbers below are the reason this lane targets mid-life pages
rather than old ones, and they do not match the intuition that older is always worse.

In [1]:
import json
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
visible = df[df["impressions_90d"] >= 100].copy().reset_index(drop=True)

age_order = ["31-90", "91-180", "181-365", "365+"]
fresh_order = ["0-30", "31-90", "91-180", "181+"]
decay = visible.groupby("age_tier")["is_declining"].agg(["size", "mean"]).reindex(age_order).round(3)
fresh = visible.groupby("freshness_tier")["is_declining"].agg(["size", "mean"]).reindex(fresh_order).round(3)
print("share of visible pages declining, by age tier")
print(decay.to_string())
print("\nshare of visible pages declining, by freshness tier")
print(fresh.to_string())
print("\nvisible base rate:", round(visible["is_declining"].mean(), 3))

share of visible pages declining, by age tier
          size   mean
age_tier             
31-90      304  0.681
91-180    8717  0.692
181-365   7650  0.606
365+      5335  0.427

share of visible pages declining, by freshness tier
                 size   mean
freshness_tier              
0-30            13735  0.583
31-90             152  0.592
91-180           8084  0.622
181+               35  0.743

visible base rate: 0.598


Two observations, both from this snapshot and both stated as observations rather than mechanisms.

Decline is a **mid-life** problem here. Pages aged 91 to 180 days are the most likely to be falling
at 0.69, against a visible base rate of 0.60, while pages past a year sit lowest at 0.43. That is
the opposite of a simple decay curve, and it has an obvious alternative explanation I cannot rule
out: my population is pages with at least 100 impressions in the last 90 days, so an old page only
appears here if it is still visible. The old cohort is a survivor cohort. The honest reading is that
among pages still worth looking at, the mid-life ones are the ones sliding, which is where I would
spend refresh slots. It is not evidence that pages get safer with age.

Freshness moves the rate far less than the paper's portfolio suggested. Going from updated in the
last 30 days to 91 to 180 days ago shifts the decline share from 0.58 to 0.62, a four point
gradient rather than a multiplier. The 181-plus bucket reads 0.74, but it holds only 35 pages, so I
report the count beside it and do not build a recommendation on it.

Now the queue itself.

In [2]:
num = ["content_age_days", "days_since_last_update", "word_count", "char_count",
       "search_volume", "competition", "cpc",
       "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
cats = ["content_type", "main_intent", "competition_level"]

def features(frame):
    f = frame.copy()
    for c in ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]:
        f["log_" + c] = np.log1p(f[c].fillna(0))
    f["has_keyword"] = f["search_volume"].notna().astype(float)
    f["has_word_count"] = f["word_count"].notna().astype(float)
    ncols = num + ["log_impressions_prev_30d", "log_clicks_prev_30d", "log_sessions_prev_30d",
                   "has_keyword", "has_word_count"]
    return pd.concat([f[ncols], f[cats].astype("object").fillna("unknown")], axis=1), ncols

Xall, ncols = features(df)
pre = ColumnTransformer([
    ("n", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value=0)),
                    ("sc", StandardScaler())]), ncols),
    ("c", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cats)])
model = Pipeline([("pre", pre), ("lr", LogisticRegression(class_weight="balanced", max_iter=2000,
                                                          C=1.0, random_state=42))])
model.fit(Xall, df["is_declining"].values)
Xvis, _ = features(visible)
visible["risk_score"] = model.predict_proba(Xvis)[:, 1]

good_tiers = ["top_3", "page_1", "striking"]
tier_ctr = visible[visible["position_tier"].isin(good_tiers)].groupby("position_tier")["ctr"].median()
visible["ctr_gap"] = visible["position_tier"].map(tier_ctr) - visible["ctr"]
stale_cut = visible["days_since_last_update"].quantile(0.75)
thin_cut = visible["word_count"].quantile(0.25)
earning_cut = visible["impressions_prev_30d"].quantile(0.60)

def triage(r):
    earning = r["impressions_prev_30d"] >= earning_cut
    if r["position_tier"] in good_tiers and r["ctr_gap"] > 0 and earning:
        return "low_ctr_for_position", "rewrite title and meta"
    if r["days_since_last_update"] >= stale_cut and earning:
        return "stale_but_earning", "refresh the content"
    if pd.notna(r["word_count"]) and r["word_count"] <= thin_cut and earning:
        return "thin_for_demand", "expand the page"
    if r["position_tier"] in ["page_3_5", "deep"]:
        return "low_visibility", "relevance work and internal links"
    return "no_clear_lever", "watch only"

triaged = visible.apply(triage, axis=1, result_type="expand")
visible["reason_code"] = triaged[0]
visible["action"] = triaged[1]

def archetype(r):
    if r["position_tier"] in ["page_3_5", "deep"] and r["impressions_prev_30d"] < earning_cut:
        return "low-demand tail"
    if r["days_since_last_update"] >= stale_cut and r["impressions_prev_30d"] >= earning_cut:
        return "neglected earner"
    if r["content_age_days"] < 90:
        return "fresh starter"
    if r["content_age_days"] > 365:
        return "old survivor"
    return "maturing asset"

visible["archetype"] = visible.apply(archetype, axis=1)

SLOTS = 50
queue = visible.sort_values("risk_score", ascending=False).head(SLOTS).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)
print("queue of", SLOTS, "slots. action mix:")
print(queue.groupby(["reason_code", "action"]).size().rename("pages").to_string())
print("\narchetype mix in the queue:")
print(queue["archetype"].value_counts().to_string())
print("\nprior-window impressions represented by the queue:",
      int(queue["impressions_prev_30d"].sum()))
print("\nreason mix across all", len(visible), "visible pages, percent:")
print((visible["reason_code"].value_counts(normalize=True) * 100).round(1).to_string())

queue of 50 slots. action mix:
reason_code           action                           
low_ctr_for_position  rewrite title and meta               15
low_visibility        relevance work and internal links    14
no_clear_lever        watch only                            1
stale_but_earning     refresh the content                  19
thin_for_demand       expand the page                       1

archetype mix in the queue:
archetype
maturing asset      27
neglected earner    23

prior-window impressions represented by the queue: 351897

reason mix across all 22006 visible pages, percent:
reason_code
no_clear_lever          48.2
low_visibility          25.3
stale_but_earning       12.3
low_ctr_for_position    12.2
thin_for_demand          2.0


The queue is not one job. Roughly two fifths of the slots are content refreshes, three tenths are
snippet rewrites, and most of the rest is relevance work on pages sitting too deep to earn clicks.
Those are different skills and different amounts of time, which is the practical reason for carrying
the reason code rather than a bare score: fifty scored pages is a number, fifty pages sorted into
three kinds of work is a plan.

Across the whole visible corpus, almost half of pages match no lever at all. That is a useful
negative result. It means a queue built from this playbook should stay short, and that pushing it
much past the top few hundred pages starts handing editors rows with no obvious thing to do.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** One content editor, once a month, opening the top of a ranked list to decide which
pages get the roughly fifty refresh slots that exist. The output is a priority order plus a starting
hypothesis for each page. It is decision-support for triage, and every row is meant to be opened and
judged, not executed.

**What the numbers actually say.** Quoted from the held-out evaluation in Weeks 5 and 6, not
recomputed here, because the queue below is produced by a model refit on all labelled pages and
scoring the same pages it trained on. That fit is the right posture for producing a queue and the
wrong one for measuring quality.

| Measurement | Value | Where from |
|---|---|---|
| Visible base rate | 0.598 | this data |
| ROC-AUC, whole clients held out | 0.625 | w06, grouped folds |
| Precision@50, six unseen clients | 0.84 | w05 lockbox |
| Same, visible-train variant | 0.76 | w05 lockbox |
| Gain from widening, 95% interval | [-0.02, +0.12] | w06 cluster bootstrap |
| Random-split inflation | +0.06 AUC | w06 before and after |

**Cost and value.** A slot costs an editor's review plus the edit. Against the 0.598 base rate,
filling fifty slots by shuffling the visible pages would land about 30 truly declining pages. The
held-out precision of 0.84 corresponds to about 42. The honest way to state the value is therefore
about twelve better-aimed slots per cycle out of fifty, measured on six held-out clients, and the
interval above says the widening part of that gain is directional rather than pinned down. The cost
of a wrong call is not symmetric: a wasted slot loses review time, while an unnecessary rewrite of a
stable page can lose the ranking it already had, which is why the no-go list below exists.

**Where it stops being valid.**

- It ranks. It does not estimate what a refresh will do. Nothing here was an experiment, so no row
  supports "refreshing this page will recover traffic".
- It only covers pages with at least 100 impressions in the trailing 90 days. Pages below that are
  outside the population by construction, and that filter uses outcome-window information.
- Per-client quality is uneven. Held-out AUC ranged from 0.44 to 0.82 across clients in Week 6, and
  one client of twenty-two ranked below chance. A client-level check comes before a client-level
  rollout.
- The reason codes read current CTR and position. In this snapshot those columns describe the same
  window the label came from, so they are diagnostics, not predictions.
- The ceiling is real. A wide search over methods and model families moved held-out AUC by
  thousandths, so a materially better queue needs better inputs, not better fitting.

In [3]:
limits = {
    "population_filter": "impressions_90d >= 100",
    "visible_pages": int(len(visible)),
    "pages_outside_population": int((df["impressions_90d"] < 100).sum()),
    "visible_base_rate": round(float(visible["is_declining"].mean()), 3),
    "held_out_auc": 0.625,
    "lockbox_p_at_50_widened": 0.84,
    "lockbox_p_at_50_visible_train": 0.76,
    "widening_gain_95_ci": [-0.02, 0.12],
    "random_split_inflation_auc": 0.06,
    "per_client_auc_range": [0.44, 0.82],
    "clients_below_chance": 1,
}
expected_random = round(SLOTS * float(visible["is_declining"].mean()), 1)
expected_model = round(SLOTS * limits["lockbox_p_at_50_widened"], 1)
print("filling", SLOTS, "slots at random from visible pages lands about", expected_random,
      "truly declining")
print("at the held-out precision of", limits["lockbox_p_at_50_widened"], "it lands about",
      expected_model)
print("difference, per cycle:", round(expected_model - expected_random, 1), "better-aimed slots")
print("\npages outside the population entirely:", limits["pages_outside_population"])

filling 50 slots at random from visible pages lands about 29.9 truly declining
at the held-out precision of 0.84 it lands about 42.0
difference, per cycle: 12.1 better-aimed slots

pages outside the population entirely: 7994


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before touching a page, the editor checks four things.** These are the failure modes the model
cannot see, and each maps to something it was never given.

1. **Is the page still what the score thinks it is?** The snapshot is trailing 90 days. A page
   rewritten last week already has a different profile.
2. **Is the decline seasonal or external?** A tax guide falling in July is behaving correctly. The
   model has no calendar and no query intent, so it cannot tell a seasonal dip from a slide.
3. **Does the reason code match what the page looks like?** The code is a threshold, not a reading.
   If `thin_for_demand` fires on a deliberately short answer page, the code is wrong and the page is
   fine.
4. **Is this page worth the ranking risk?** Rewriting a page that still ranks can cost the position
   it holds. For `low_ctr_for_position` the safe version is a snippet change, not a rebuild.

**The no-go list. None of the following should be automated on this score.**

| Never automate | Why |
|---|---|
| Publishing a rewrite without a human read | The score ranks risk. It has no view of quality, accuracy or brand voice. |
| Bulk edits across a whole client | Per-client quality ranges from 0.44 to 0.82 AUC. One client ranked below chance. |
| Acting on pages outside the population | Below 100 impressions the model was never trained to rank, and the label is mostly noise there. |
| Deleting, deindexing or consolidating pages | A decline score is not evidence a page has no value. Removal is irreversible and needs its own analysis. |
| Reporting the score to clients as a forecast | It is a ranking aid measured at 0.625 AUC, not a prediction of traffic. |
| Using the score in anyone's performance review | It was built to order a worklist, and the label is a defined proxy, not a measure of an editor's work. |
| Letting the score pick its own retraining label | The proxy would then reinforce itself, and the queue would drift toward whatever it already believes. |

The last row is the one that would quietly break the system. If refreshed pages are treated as
successes and fed back as labels, the model learns which pages were chosen rather than which pages
declined. Keeping the label measured from traffic, independent of what the queue recommended, is
what keeps the next version honest.

In [4]:
per_client = visible.groupby("client_id").agg(
    pages=("content_id", "size"),
    declining=("is_declining", "mean"),
    queued=("risk_score", lambda s: 0)).round(3)
per_client["queued"] = queue["client_id"].value_counts().reindex(per_client.index).fillna(0).astype(int)
concentration = per_client.sort_values("queued", ascending=False).head(5)
print("queue concentration, top clients by slots taken:")
print(concentration.to_string())
print("\nclients represented in the queue:", int((per_client["queued"] > 0).sum()),
      "of", len(per_client))
print("largest share of the queue taken by one client:",
      round(float(per_client["queued"].max()) / SLOTS, 2))

queue concentration, top clients by slots taken:
                   pages  declining  queued
client_id                                  
client_6208ef0f77   3568      0.641      20
client_7f2253d7e2   1019      0.956      10
client_f369cb89fc    935      0.677       7
client_3fdba35f04   2149      0.845       5
client_19581e27de   6579      0.489       4

clients represented in the queue: 9 of 30
largest share of the queue taken by one client: 0.4


A queue that hands most of its slots to one client is a review flag rather than a result. The
concentration above is worth reading before the list goes out: if one client dominates, that is
usually its size showing through, and a per-client cap is the fix. This is a policy decision for a
human, which is why it sits here rather than in the scoring code.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The model is a snapshot fit, so the question is not whether it drifts but when someone notices.
Four checks, cheap enough to run each cycle, with the number that should trigger a look.

| Check | Run | Trigger | What it probably means |
|---|---|---|---|
| Base rate of the label | monthly | moves outside 0.55 to 0.65 | the population or the trend definition shifted, and every precision number needs restating |
| Realised precision of worked slots | each cycle | fewer than 30 of 50 worked pages were truly declining | the queue has stopped beating a coin flip against the base rate |
| Score distribution | monthly | median risk score moves by more than 0.05 | inputs shifted under the model even if the label did not |
| Feature missingness | monthly | `has_keyword` or `has_word_count` share moves by more than 10 points | an upstream feed changed, and zero-filled columns are quietly encoding that change |

**Retrain triggers, in order of what I would actually do.** Any single trigger means investigate,
not retrain. Retraining a drifted model on drifted data usually hides the problem.

1. Two consecutive cycles below the realised-precision floor, with the base rate stable. Refit on
   the newest snapshot and re-run the Week-6 audit before shipping.
2. A base-rate move outside the band. Do not refit yet. Re-derive the label window first, because a
   moved base rate usually means the definition or the population changed.
3. A new client onboarded. Do not assume the model transfers. Score their pages, hold their client
   out, and read the per-client AUC before the queue is used for them.
4. Any change to the upstream columns. Re-run the leakage audit, because a renamed or re-windowed
   traffic column is exactly how a clean feature set turns leaky.

Whatever triggers a retrain, the release check is the same one from Week 6: whole clients held out,
base rate printed next to every metric, and the sealed clients read once.

In [5]:
monitor = {
    "base_rate_band": [0.55, 0.65],
    "current_base_rate": round(float(visible["is_declining"].mean()), 3),
    "realised_precision_floor_per_50": 30,
    "current_median_risk": round(float(visible["risk_score"].median()), 3),
    "has_keyword_share": round(float(visible["search_volume"].notna().mean()), 3),
    "has_word_count_share": round(float(visible["word_count"].notna().mean()), 3),
}
print("baseline values to compare future cycles against:")
for k, v in monitor.items():
    print(f"  {k}: {v}")

baseline values to compare future cycles against:
  base_rate_band: [0.55, 0.65]
  current_base_rate: 0.598
  realised_precision_floor_per_50: 30
  current_median_risk: 0.598
  has_keyword_share: 0.974
  has_word_count_share: 0.702


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Three kinds of artifact, treated differently on purpose. The ranked queue is data, so it is written
to `work/outputs/` and stays out of git. The figures are for the paper, so they go to
`work/figures/` and are committed. The metrics file is the receipt that every number in the paper
can be traced back to, so it is committed too.

In [6]:
Path("../outputs").mkdir(exist_ok=True)
Path("../figures").mkdir(exist_ok=True)

queue_cols = ["rank", "content_id", "client_id", "risk_score", "reason_code", "action",
              "archetype", "content_type", "age_tier", "freshness_tier", "position_tier",
              "impressions_prev_30d", "days_since_last_update", "word_count"]
queue_out = queue[queue_cols].copy()
queue_out["risk_score"] = queue_out["risk_score"].round(4)
queue_out.to_csv("../outputs/refresh_queue.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, table, order, title in [
        (axes[0], decay, age_order, "by age tier"),
        (axes[1], fresh, fresh_order, "by freshness tier")]:
    ax.bar(order, table["mean"].values, color="#4C72B0")
    ax.axhline(visible["is_declining"].mean(), color="#C44E52", linestyle="--", linewidth=1)
    ax.set_ylim(0, 0.85)
    ax.set_title("Share of visible pages declining, " + title)
    ax.set_ylabel("share declining")
    for i, (n, m) in enumerate(zip(table["size"].values, table["mean"].values)):
        ax.text(i, m + 0.02, f"n={int(n)}", ha="center", fontsize=8)
fig.suptitle("Dashed line is the visible base rate of 0.598", fontsize=9, y=0.02)
fig.tight_layout()
fig.savefig("../figures/decline_by_age_and_freshness.png", dpi=150, bbox_inches="tight")
plt.close(fig)

mix = queue["action"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(mix.index, mix.values, color="#55A868")
ax.set_xlabel("slots in the queue of " + str(SLOTS))
ax.set_title("What the top of the queue asks an editor to do")
for i, v in enumerate(mix.values):
    ax.text(v + 0.3, i, str(int(v)), va="center", fontsize=9)
fig.tight_layout()
fig.savefig("../figures/queue_action_mix.png", dpi=150, bbox_inches="tight")
plt.close(fig)

metrics = {
    "queue": {
        "slots": SLOTS,
        "visible_pages_scored": int(len(visible)),
        "prior_window_impressions_in_queue": int(queue["impressions_prev_30d"].sum()),
        "clients_represented": int(queue["client_id"].nunique()),
        "action_mix": {k: int(v) for k, v in queue["action"].value_counts().items()},
        "archetype_mix": {k: int(v) for k, v in queue["archetype"].value_counts().items()},
    },
    "reason_code_share_all_visible": {
        k: round(float(v), 3) for k, v in visible["reason_code"].value_counts(normalize=True).items()},
    "decay_by_age_tier": {k: {"pages": int(r["size"]), "share_declining": float(r["mean"])}
                          for k, r in decay.iterrows()},
    "freshness_by_tier": {k: {"pages": int(r["size"]), "share_declining": float(r["mean"])}
                          for k, r in fresh.iterrows()},
    "thresholds": {
        "stale_days_p75": float(round(stale_cut, 1)),
        "thin_word_count_p25": float(round(thin_cut, 1)),
        "earning_impressions_p60": float(round(earning_cut, 1)),
    },
    "limits": limits,
    "monitoring": monitor,
}
with open("../outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("wrote ../outputs/refresh_queue.csv        rows:", len(queue_out), "(data, stays out of git)")
print("wrote ../figures/decline_by_age_and_freshness.png  (committed)")
print("wrote ../figures/queue_action_mix.png              (committed)")
print("wrote ../outputs/playbook_metrics.json             (committed receipt)")
print("\nqueue preview:")
print(queue_out.head(8)[["rank", "risk_score", "reason_code", "action", "archetype"]].to_string(index=False))

wrote ../outputs/refresh_queue.csv        rows: 50 (data, stays out of git)
wrote ../figures/decline_by_age_and_freshness.png  (committed)
wrote ../figures/queue_action_mix.png              (committed)
wrote ../outputs/playbook_metrics.json             (committed receipt)

queue preview:
 rank  risk_score          reason_code                            action        archetype
    1      0.9672       no_clear_lever                        watch only   maturing asset
    2      0.9595 low_ctr_for_position            rewrite title and meta   maturing asset
    3      0.9315 low_ctr_for_position            rewrite title and meta   maturing asset
    4      0.9308    stale_but_earning               refresh the content neglected earner
    5      0.9235 low_ctr_for_position            rewrite title and meta   maturing asset
    6      0.9215       low_visibility relevance work and internal links   maturing asset
    7      0.9184       low_visibility relevance work and internal links   maturi

The CSV carries pseudonymous ids only, the same ones the starter data ships with, and no page
titles, URLs or client names. It is regenerated by running this notebook, which is why it does not
need to live in git.

What the paper takes from here: the two figures, the metrics file for any number quoted, and the
framing that a decline score becomes useful only once it carries a reason, a limit and a review
step.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime, Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit the repo URL on the card